# Preprocessing Practice - bike_resale.csv

A resale website has data about used bikes. Turn this file into a clean numeric table.

We want to predict whether a bike gets sold in the first week (`sold_in_week`).

**Target shape at the end: (450, 16)**

In [1]:
import numpy as np
import pandas as pd
import sklearn

In [3]:
df=pd.read_csv('bike_resale.csv')
df.head()

,bike_id,brand,age_months,km_driven,owner_count,condition,mileage_kmpl,service_count,insurance,price_thousand,sold_in_week
0,B4543,TVS,25,14455.0,3,Excellent,40.6,NaN,Yes,41.6,No
1,B4392,TVS,43,26304.0,2,Good,51.2,1.0,Yes,24.1,No
2,B4235,Honda,38,6421.0,1,Good,65.0,NaN,No,48.2,No
3,B4319,Hero,14,6531.0,1,Good,NaN,NaN,Yes,58.7,No
4,B4284,TVS,115,80000.0,1,Good,61.8,0.0,No,12.0,Yes


---
## Step 1 - Load and look

Read the file and check its rows, shape and column types.

Expected shape: `(450, 11)`

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 450 entries, 0 to 449
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   bike_id         450 non-null    object 
 1   brand           450 non-null    object 
 2   age_months      450 non-null    int64  
 3   km_driven       441 non-null    float64
 4   owner_count     450 non-null    int64  
 5   condition       450 non-null    object 
 6   mileage_kmpl    424 non-null    float64
 7   service_count   376 non-null    float64
 8   insurance       450 non-null    object 
 9   price_thousand  450 non-null    float64
 10  sold_in_week    450 non-null    object 
dtypes: float64(4), int64(2), object(5)
memory usage: 38.8+ KB


In [7]:
df.describe()

,age_months,km_driven,owner_count,mileage_kmpl,service_count,price_thousand
count,450.000000,441.000000,450.000000,424.000000,376.000000,450.000000
mean,60.115556,35447.324263,1.524444,47.882075,2.300532,32.669778
std,33.513152,22666.829259,0.712959,9.609344,1.428323,27.750581
min,3.000000,475.000000,1.000000,22.000000,0.000000,12.000000
25%,31.500000,15268.000000,1.000000,41.775000,1.000000,12.000000
50%,60.000000,32626.000000,1.000000,48.000000,2.000000,20.600000
75%,88.750000,52773.000000,2.000000,54.925000,3.000000,47.400000
max,120.000000,80000.000000,3.000000,72.000000,5.000000,172.300000


In [8]:
num_cols = ['age_months', 'km_driven', 'owner_count', 'mileage_kmpl', 'service_count']
cat_cols=['brand','condition','insurance']
print(num_cols)
print(cat_cols)

['age_months', 'km_driven', 'owner_count', 'mileage_kmpl', 'service_count']
['brand', 'condition', 'insurance']


---
## Step 2 - Separate numeric and categorical columns

Make two lists: number columns and word columns.

Leave out the id column and both answer columns (`price_thousand`, `sold_in_week`).

Expected: `5` numeric, `3` categorical

---
## Step 3 - Split into X and y

Expected: X `(450, 8)`, y `(450,)`

In [10]:
x=df.drop(columns=['bike_id','price_thousand','sold_in_week'])
y=df['sold_in_week']
print(x.shape,y.shape)

(450, 8) (450,)


---
## Step 4 - Handle missing values

Count the missing values, then fill each numeric column with its own mean.

Expected: every count becomes `0`

In [11]:
df.isna().sum()

bike_id            0
brand              0
age_months         0
km_driven          9
owner_count        0
condition          0
mileage_kmpl      26
service_count     74
insurance          0
price_thousand     0
sold_in_week       0
dtype: int64

In [13]:
for col in num_cols:
    x[col]=x[col].fillna(x[col].mean())

---
## Step 5 - Encode categorical columns

Use `OneHotEncoder` on the word columns.

Expected shape: `(450, 11)`

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encdoer=OneHotEncoder()
encoded=encdoer.fit_transform(x[cat_cols])
print(encoded.shape)


(450, 11)


In [15]:
encdoer.get_feature_names_out()

array(['brand_Bajaj', 'brand_Hero', 'brand_Honda', 'brand_Royal Enfield',
       'brand_TVS', 'brand_Yamaha', 'condition_Excellent',
       'condition_Fair', 'condition_Good', 'insurance_No',
       'insurance_Yes'], dtype=object)

---
## Step 6 - Scale numeric columns

Compare `owner_count` and `km_driven` before and after `StandardScaler`.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled=scaler.fit_transform(x[num_cols])
print(scaled.shape)

(450, 5)


In [17]:
scaled_df=pd.DataFrame(scaled,columns=num_cols)
scaled_df.head()

,age_months,km_driven,owner_count,mileage_kmpl,service_count
0,-1.048980,-0.936590,2.071927,-0.781623,3.405920e-16
1,-0.511280,-0.407937,0.667760,0.356130,-9.974364e-01
2,-0.660641,-1.295034,-0.736408,1.837357,3.405920e-16
3,-1.377575,-1.290126,-0.736408,0.000000,3.405920e-16
4,1.639521,1.987755,-0.736408,1.493884,-1.764381e+00


---
## Step 7 - Both together with ColumnTransformer

Expected shape: `(450, 16)`

In [18]:
from sklearn.compose import ColumnTransformer

preprocessor=ColumnTransformer([
    ('cat',OneHotEncoder(),cat_cols),
    ('nums',StandardScaler(),num_cols)
])
X_processed=preprocessor.fit_transform(x)
print(X_processed.shape)

(450, 16)
